# Section 4.4 — Expert Withdrawal
## Task 2: three-class person counting

Part A is the multiclass cutoff experiment exactly as written: expert-only
supervising trajectory, affine parameters $(\hat b_i, \hat c_i,
\hat\sigma_i^2)$ fitted by least squares during training, camera withdrawn at
$k_c = 1250$, microphones inverted and combined by inverse-variance weighting
thereafter.

Part B adds no computation to the experiment. It reads `Xki_all`, `Xki_expert`
and `Xd` and reports the expert baseline on each segment, the class
distribution of each segment, and per-class recall before and after the cutoff.

## Part A — the experiment

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================================
# MULTICLASS CUTOFF EXPERIMENT
# ==========================================================
# Purpose: extend the binary expert-cutoff experiment to the
# multiclass (0/1/2 person counting) setting. During TRAIN phase,
# the camera + 2 mics fuse together and the mics learn their affine
# parameters (b, c, sigma^2) from a chosen teacher signal. At K_CUT,
# the camera is removed and the mics operate alone using their frozen
# learned parameters.
#
# TEACHER_MODE controls what the mics learn from during TRAIN:
#   "gt"     : ground truth Xd (upper bound, requires labels)
#   "expert" : expert-only KF output (autonomous, no GT labels needed)
#   "fused"  : fused model's own output (paper's approach, circular risk)
# ==========================================================

# ------------------ STYLE ------------------
sns.set(style="whitegrid")
sns.set_palette("deep")
sns.set(font="DejaVu Sans")
np.random.seed(1)

# ==========================
# SETTINGS -- adjust these
# ==========================
CSV_PATH       = "../data/person_counting_2163.csv"
K_CUT          = 1250          # camera removed after this sample
TEACHER_MODE   = "expert"          # "gt", "expert", or "fused"
eps            = 1e-9

# ==========================
# LOAD DATA
# ==========================
df = pd.read_csv(CSV_PATH)
needed = ["human_count", "mouse_event", "dBFS_mic1", "dBFS_mic2"]
df = df[needed].dropna().reset_index(drop=True)
df = df.rename(columns={"human_count": "human", "mouse_event": "mouse"})

Np = len(df)
if K_CUT >= Np - 1:
    raise ValueError(f"K_CUT={K_CUT} must be < Np-1={Np-1}")

# ==========================
# MULTICLASS GROUND TRUTH FROM MOUSE EVENTS
# ==========================
m = df["mouse"].astype(int).to_numpy()
count_gt = np.zeros(len(m), dtype=int)
active_class = 0
for k, event in enumerate(m):
    if event == 1:
        active_class = 0 if active_class == 1 else 1
    elif event == 2:
        active_class = 0 if active_class == 2 else 2
    count_gt[k] = active_class
df["count_gt"] = count_gt

# ==========================
# SIGNALS
# ==========================
Xd  = df["count_gt"].to_numpy().astype(int)   # true GT (0/1/2)
y1  = df["human"].astype(float).to_numpy()     # expert camera (0/1/2 count)
y0  = df["dBFS_mic1"].astype(float).to_numpy() # inexpert mic 1 (NOT normalized)
y2  = df["dBFS_mic2"].astype(float).to_numpy() # inexpert mic 2 (NOT normalized)

CLASS_LEVELS = np.array([0, 1, 2])

def quantize_count(x):
    return int(CLASS_LEVELS[np.argmin(np.abs(CLASS_LEVELS - float(x)))])

# ==========================
# KF PARAMETERS
# ==========================
sig1 = 1.0
c1   = 1.0

# ==========================
# AFFINE PARAMETER LEARNING
# ==========================
def fit_affine_sensor(y, x, prev_b, prev_c):
    """
    Fits y ~ b + c*X via ordinary least squares.
    Falls back to previous params if rank-deficient (only one class seen).
    Returns (b_hat, c_hat, sigma2_hat).
    """
    x = np.asarray(x, dtype=float).reshape(-1)
    y = np.asarray(y, dtype=float).reshape(-1)
    H = np.column_stack([x, np.ones_like(x)])
    if np.linalg.matrix_rank(H) < 2:
        residual = y - (prev_b + prev_c * x)
        return prev_b, prev_c, max(float(np.var(residual)), 1e-6)
    theta  = np.linalg.pinv(H) @ y
    c_hat  = float(theta[0])
    b_hat  = float(theta[1])
    residual = y - (b_hat + c_hat * x)
    return b_hat, c_hat, max(float(np.var(residual)), 1e-6)

# ==========================
# STORAGE
# ==========================
Xki_all      = np.zeros(Np, dtype=int)   # final fused/deploy classification
Xki_expert   = np.zeros(Np, dtype=int)   # expert-only KF (for "expert" teacher + comparison)

bh0_hist = np.zeros(Np); ch0_hist = np.zeros(Np); rh0_hist = np.zeros(Np)
bh2_hist = np.zeros(Np); ch2_hist = np.zeros(Np); rh2_hist = np.zeros(Np)

# ==========================
# INITIALIZATION
# ==========================

# Expert-only KF (runs throughout for comparison and optional teacher)
X_exp = np.array([0.0])
P_exp = np.eye(1)
Q_exp = 10 * np.eye(1)
R_exp = np.array([[sig1**2]])
C_exp = np.array([[c1]])
A_exp = np.eye(1)

# Fused KF with augmented state [X, 1]^T
X_fused = np.array([0.0, 1.0])
P_fused = np.diag([1.0, 0.0])
Q_fused = np.diag([10.0, 0.0])
A_fused = np.eye(2)

# Initial inexpert parameters (reasonable defaults)
bh0, ch0, rh0 = 0.0, 1.0, 1.0
bh2, ch2, rh2 = 0.0, 1.0, 1.0

# Freeze storage
cut_done      = False
c_frozen_0    = None;  b_frozen_0    = None;  sig2_frozen_0 = None
c_frozen_2    = None;  b_frozen_2    = None;  sig2_frozen_2 = None

print(f"Teacher mode: {TEACHER_MODE}")
print(f"Camera cut at k={K_CUT} (of {Np} total samples)")
print(f"Class distribution: {np.bincount(Xd)}")

Teacher mode: expert
Camera cut at k=1250 (of 2163 total samples)
Class distribution: [860 559 744]


In [2]:
# ==========================
# MAIN LOOP
# ==========================
for k in range(Np):

    # ---- Expert-only KF (always runs) ----
    Xm_e  = A_exp @ X_exp.reshape(-1, 1)
    Pm_e  = A_exp @ P_exp @ A_exp.T + Q_exp
    res_e = y1[k] - (C_exp @ Xm_e).flatten()
    K_e   = Pm_e @ C_exp.T @ np.linalg.inv(C_exp @ Pm_e @ C_exp.T + R_exp)
    X_exp = (Xm_e + K_e @ res_e.reshape(-1, 1)).flatten()
    P_exp = (np.eye(1) - K_e @ C_exp) @ Pm_e
    Xki_expert[k] = quantize_count(X_exp[0])

    # ================================================
    # TRAIN PHASE: k <= K_CUT
    # ================================================
    if k <= K_CUT:

        rh0 = max(float(rh0), 1e-6)
        rh2 = max(float(rh2), 1e-6)

        R_all = np.array([
            [sig1**2, 0.0, 0.0],
            [0.0,     rh0, 0.0],
            [0.0,     0.0, rh2]
        ])
        C_all = np.array([
            [c1,  0.0],
            [ch0, bh0],
            [ch2, bh2]
        ])

        Xm_f = A_fused @ X_fused.reshape(-1, 1)
        Pm_f = A_fused @ P_fused @ A_fused.T + Q_fused

        y_vec = np.array([y1[k], y0[k], y2[k]]).reshape(-1, 1)
        res_f = y_vec - C_all @ Xm_f

        K_f     = Pm_f @ C_all.T @ np.linalg.inv(C_all @ Pm_f @ C_all.T + R_all)
        X_fused = (Xm_f + K_f @ res_f).flatten()
        P_fused = (np.eye(2) - K_f @ C_all) @ Pm_f
        X_fused[1] = 1.0   # keep augmented constant pinned

        Xki_all[k] = quantize_count(X_fused[0])

        # ---- Teacher signal selection ----
        if TEACHER_MODE == "gt":
            teacher = Xd[:k+1].astype(float)
        elif TEACHER_MODE == "expert":
            teacher = Xki_expert[:k+1].astype(float)
        else:  # "fused"
            teacher = Xki_all[:k+1].astype(float)

        # ---- Update affine parameters ----
        bh0, ch0, rh0 = fit_affine_sensor(y0[:k+1], teacher, bh0, ch0)
        bh2, ch2, rh2 = fit_affine_sensor(y2[:k+1], teacher, bh2, ch2)

        bh0_hist[k] = bh0;  ch0_hist[k] = ch0;  rh0_hist[k] = rh0
        bh2_hist[k] = bh2;  ch2_hist[k] = ch2;  rh2_hist[k] = rh2

        # ---- Freeze at the cutoff point ----
        if (not cut_done) and (k == K_CUT):
            cut_done      = True
            c_frozen_0    = float(ch0);  b_frozen_0    = float(bh0)
            sig2_frozen_0 = float(rh0)
            c_frozen_2    = float(ch2);  b_frozen_2    = float(bh2)
            sig2_frozen_2 = float(rh2)
            print(f"\n=== CUT at k={K_CUT} (teacher={TEACHER_MODE}) ===")
            print(f"Frozen mic 1: b={b_frozen_0:.4f}, c={c_frozen_0:.4f}, sigma2={sig2_frozen_0:.4f}")
            print(f"Frozen mic 2: b={b_frozen_2:.4f}, c={c_frozen_2:.4f}, sigma2={sig2_frozen_2:.4f}")
            print("DEPLOY starts: camera OFF, two mics only via frozen affine model.\n")

    # ================================================
    # DEPLOY PHASE: k > K_CUT
    # ================================================
    else:
        # Invert the learned affine model for each mic:
        #   y = b + cX  =>  X_hat = (y - b) / c
        x_hat_0 = (y0[k] - b_frozen_0) / (c_frozen_0 + eps)
        x_hat_2 = (y2[k] - b_frozen_2) / (c_frozen_2 + eps)

        # Weighted average using learned noise variances (lower variance = more trust)
        w0 = 1.0 / (sig2_frozen_0 + eps)
        w2 = 1.0 / (sig2_frozen_2 + eps)
        x_hat = (w0 * x_hat_0 + w2 * x_hat_2) / (w0 + w2)

        Xki_all[k] = quantize_count(x_hat)

        # Store frozen params for plotting continuity
        bh0_hist[k] = b_frozen_0;  ch0_hist[k] = c_frozen_0;  rh0_hist[k] = sig2_frozen_0
        bh2_hist[k] = b_frozen_2;  ch2_hist[k] = c_frozen_2;  rh2_hist[k] = sig2_frozen_2


=== CUT at k=1250 (teacher=expert) ===
Frozen mic 1: b=-34.5672, c=4.0882, sigma2=66.7275
Frozen mic 2: b=-30.1514, c=3.8778, sigma2=43.8918
DEPLOY starts: camera OFF, two mics only via frozen affine model.



In [3]:
# ==========================
# RESULTS
# ==========================
err_all    = (Xki_all    != Xd).astype(int)
err_expert = (Xki_expert != Xd).astype(int)

err_train  = 100 * np.mean(err_all[:K_CUT+1])
err_deploy = 100 * np.mean(err_all[K_CUT+1:])
err_expert_overall = 100 * np.mean(err_expert)

cum_all    = 100.0 * np.cumsum(err_all)    / (np.arange(Np) + 1)
cum_expert = 100.0 * np.cumsum(err_expert) / (np.arange(Np) + 1)

deploy_err = err_all[K_CUT+1:]
cum_deploy = 100.0 * np.cumsum(deploy_err) / (np.arange(len(deploy_err)) + 1)

print(f"Expert-only  (full run)       : {err_expert_overall:.2f}%")
print(f"Fused TRAIN  (k=0 to {K_CUT})  : {err_train:.2f}%")
print(f"Deploy ONLY  (k={K_CUT+1} to end): {err_deploy:.2f}%")

# Per-class error in deploy phase
print("\nPer-class error in DEPLOY phase:")
for cls in [0, 1, 2]:
    mask = (Xd[K_CUT+1:] == cls)
    if mask.sum() == 0:
        print(f"  Class {cls}: no samples in deploy window")
        continue
    cls_err = 100 * np.mean(err_all[K_CUT+1:][mask])
    print(f"  Class {cls}: {cls_err:.2f}% error ({mask.sum()} samples)")

Expert-only  (full run)       : 44.24%
Fused TRAIN  (k=0 to 1250)  : 36.29%
Deploy ONLY  (k=1251 to end): 22.70%

Per-class error in DEPLOY phase:
  Class 0: 1.20% error (332 samples)
  Class 1: 52.03% error (246 samples)
  Class 2: 22.46% error (334 samples)


In [4]:
# ---------- sensitivity of the counting result to the cutoff index ----------
# Part A logic unchanged; only k_cut varies.

def run_cutoff_mc(k_cut):
    Xk = np.zeros(Np, dtype=int)
    Xf = np.array([0.0, 1.0])
    Pf = np.diag([1.0, 0.0])
    Qf = np.diag([10.0, 0.0])
    Af = np.eye(2)
    b0, c0, r0 = 0.0, 1.0, 1.0
    b2, c2_, r2 = 0.0, 1.0, 1.0
    frozen = None

    for k in range(Np):
        if k <= k_cut:
            r0 = max(float(r0), 1e-6); r2 = max(float(r2), 1e-6)
            R = np.array([[sig1**2, 0.0, 0.0], [0.0, r0, 0.0], [0.0, 0.0, r2]])
            C = np.array([[c1, 0.0], [c0, b0], [c2_, b2]])

            Xm = Af @ Xf.reshape(-1, 1)
            Pm = Af @ Pf @ Af.T + Qf
            yv = np.array([y1[k], y0[k], y2[k]]).reshape(-1, 1)
            Kf = Pm @ C.T @ np.linalg.inv(C @ Pm @ C.T + R)
            Xf = (Xm + Kf @ (yv - C @ Xm)).flatten()
            Pf = (np.eye(2) - Kf @ C) @ Pm
            Xf[1] = 1.0
            Xk[k] = quantize_count(Xf[0])

            teacher = Xki_expert[:k+1].astype(float)
            b0, c0, r0  = fit_affine_sensor(y0[:k+1], teacher, b0, c0)
            b2, c2_, r2 = fit_affine_sensor(y2[:k+1], teacher, b2, c2_)
            if k == k_cut:
                frozen = (b0, c0, r0, b2, c2_, r2)
        else:
            fb0, fc0, fr0, fb2, fc2, fr2 = frozen
            x0 = (y0[k] - fb0) / (fc0 + eps)
            x2 = (y2[k] - fb2) / (fc2 + eps)
            w0, w2 = 1.0 / (fr0 + eps), 1.0 / (fr2 + eps)
            Xk[k] = quantize_count((w0 * x0 + w2 * x2) / (w0 + w2))

    dep = slice(k_cut + 1, Np)
    rec = [100.0 * np.mean(Xk[dep][Xd[dep] == cl] == cl)
           if np.sum(Xd[dep] == cl) else np.nan for cl in [0, 1, 2]]
    return (100 * np.mean(Xk[:k_cut+1] != Xd[:k_cut+1]),
            100 * np.mean(Xk[dep] != Xd[dep]),
            100 * np.mean(Xki_expert[dep] != Xd[dep]),
            rec, frozen,
            int(np.sum(Xki_expert[:k_cut+1] == 2)),   # times the teacher said "2"
            int(np.sum(Xd[:k_cut+1] == 2)))           # times it actually was

grid = [200, 300, 400, 600, 800, 1000, 1250, 1500, 1800]
rows = []
print(f"{'k_c':>6}{'n_dep':>7}{'train':>9}{'deploy':>9}{'expert':>9}"
      f"{'rec 0':>8}{'rec 1':>8}{'rec 2':>8}{'c_1':>9}{'c_2':>9}"
      f"{'T=2':>7}{'true 2':>8}")
for kc in grid:
    tr_e, dp_e, ex_e, rec, fz, n_t2, n_x2 = run_cutoff_mc(kc)
    rows.append((kc, dp_e, ex_e, rec[2]))
    print(f"{kc:>6}{Np-kc-1:>7}{tr_e:>8.2f}%{dp_e:>8.2f}%{ex_e:>8.2f}%"
          f"{rec[0]:>8.1f}{rec[1]:>8.1f}{rec[2]:>8.1f}"
          f"{fz[1]:>9.4f}{fz[4]:>9.4f}{n_t2:>7}{n_x2:>8}")

kcs = [r[0] for r in rows]
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(kcs, [r[2] for r in rows], "--", color="gray", marker="o",
        markersize=4, label="Expert alone, deployment segment")
ax.plot(kcs, [r[1] for r in rows], color="steelblue", marker="o",
        markersize=4, label="Microphones alone, deployment segment")
ax.plot(kcs, [r[3] for r in rows], color="darkorange", marker="s",
        markersize=4, label="Two-person recall, deployment")
ax.set_xlabel("Cutoff index $k_c$")
ax.set_ylabel("Percent [%]")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("ch5_counting_kc_sweep.pdf", bbox_inches="tight")
plt.show()

   k_c  n_dep    train   deploy   expert   rec 0   rec 1   rec 2      c_1      c_2    T=2  true 2


   200   1962   44.28%   21.20%   43.83%    99.5    32.0    90.1   3.8222   2.1777     23      85


   300   1862   41.86%   19.12%   43.93%    99.2    39.8    90.0   3.7377   2.7361     27     107


   400   1762   39.65%   17.54%   43.81%    99.3    36.9    92.6   3.4165   2.5026     28     107


   600   1562   38.60%   17.80%   44.11%    99.2    42.4    89.3   3.9575   3.1222     46     183


...

  1000   1162   37.96%   24.18%   44.58%    98.9    43.5    76.6   3.8666   3.7365     84     364


  1250    912   36.29%   22.70%   46.49%    98.8    48.0    77.5   4.0882   3.8778     97     410


  1500    662   35.11%   23.87%   50.91%    98.5    50.3    78.3   4.2518   4.0967    112     481


  1800    362   36.54%   27.07%   51.93%    98.0    44.8    75.5   4.0212   3.8905    138     589


/tmp/ipykernel_993/831561875.py:76: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# ==========================
# CONFUSION MATRICES
# ==========================
def cm3(y_true, y_pred):
    cm = np.zeros((3, 3), dtype=int)
    for t, p in zip(y_true.astype(int), y_pred.astype(int)):
        if 0 <= t <= 2 and 0 <= p <= 2:
            cm[t, p] += 1
    return cm

cm_train_expert = cm3(Xd[:K_CUT+1], Xki_expert[:K_CUT+1])
cm_train_fused  = cm3(Xd[:K_CUT+1], Xki_all[:K_CUT+1])
cm_deploy       = cm3(Xd[K_CUT+1:], Xki_all[K_CUT+1:])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sns.heatmap(cm_train_expert, annot=True, fmt="d", cbar=False, ax=axes[0])
axes[0].set_title(f"TRAIN: Expert only\n(k=0 to {K_CUT})")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")

sns.heatmap(cm_train_fused, annot=True, fmt="d", cbar=False, ax=axes[1])
axes[1].set_title(f"TRAIN: Fused (teacher={TEACHER_MODE})\n(k=0 to {K_CUT})")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True")

sns.heatmap(cm_deploy, annot=True, fmt="d", cbar=False, ax=axes[2])
axes[2].set_title(f"DEPLOY: Mics only (frozen params)\n(k={K_CUT+1} to end)")
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("True")

plt.tight_layout()
plt.savefig("multiclass_cutoff_confusion.pdf")
plt.show()

/tmp/ipykernel_993/2676730067.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
TRAIN = slice(0, K_CUT + 1)
DEPLOY = slice(K_CUT + 1, Np)

# ---------- Figures for Section 4.4, as PDF ----------
# Reads Xd, Xki_all, Xki_expert only. Part A is untouched.

# 1) confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
panels = [(cm3(Xd[TRAIN],  Xki_expert[TRAIN]), "Expert alone, training"),
          (cm3(Xd[TRAIN],  Xki_all[TRAIN]),    "Fused, training"),
          (cm3(Xd[DEPLOY], Xki_all[DEPLOY]),   "Microphones alone, deployment")]
for ax, (cm, ttl) in zip(axes, panels):
    sns.heatmap(cm, annot=True, fmt="d", cbar=False, ax=ax, cmap="Blues")
    ax.set_title(ttl, fontsize=10)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout()
plt.savefig("ch5_counting_confusion.pdf", bbox_inches="tight")
plt.show()

# 2) cumulative error with the expert baseline
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(cum_expert, linestyle="--", color="gray", linewidth=1.6,
        label=f"Expert alone ({cum_expert[-1]:.2f}%)")
ax.plot(cum_all, color="steelblue", linewidth=2,
        label=f"Network ({cum_all[-1]:.2f}%)")
ax.axvline(K_CUT, linestyle="--", color="firebrick", linewidth=1.6,
           label=f"Expert withdrawn, $k_c$ = {K_CUT}")
ax.set_xlabel("Time step $k$")
ax.set_ylabel("Cumulative error [%]")
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.savefig("ch5_counting_cumulative.pdf", bbox_inches="tight")
plt.show()

/tmp/ipykernel_993/48939690.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/tmp/ipykernel_993/48939690.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# ==========================
# PLOT 1: Overall cumulative error
# ==========================
plt.figure(figsize=(11, 4))
plt.plot(cum_expert, label=f"Expert only: {err_expert_overall:.2f}%",
         linestyle="--", color="gray")
plt.plot(cum_all, label=f"Fused/Deploy (teacher={TEACHER_MODE}): {cum_all[-1]:.2f}%",
         color="steelblue", linewidth=2)
plt.axvline(K_CUT, linestyle="--", color="firebrick", linewidth=2,
            label=f"Camera cut @ k={K_CUT}")
plt.ylabel("Cumulative Percent Error [%]")
plt.xlabel("Time step k")
plt.title(f"Multiclass Cutoff: Overall (teacher={TEACHER_MODE})")
plt.legend(loc="upper right")
plt.tight_layout()
plt.savefig("multiclass_cutoff_overall.png", dpi=150)
plt.show()

# ==========================
# PLOT 2: Deploy phase only
# ==========================
plt.figure(figsize=(11, 4))
plt.plot(np.arange(K_CUT+1, Np), cum_deploy,
         label=f"Mics only (frozen): {err_deploy:.2f}%",
         color="steelblue", linewidth=2)
plt.axvline(K_CUT, linestyle="--", color="firebrick", linewidth=2,
            label=f"Camera cut @ k={K_CUT}")
plt.ylabel("Cumulative Percent Error [%]")
plt.xlabel("Time step k")
plt.title(f"DEPLOY phase only: mics alone (teacher={TEACHER_MODE}, cut @ k={K_CUT})")
plt.legend(loc="upper right")
plt.tight_layout()
plt.savefig("multiclass_cutoff_deploy.png", dpi=150)
plt.show()

# ==========================
# PLOT 3: Before vs After bar chart
# ==========================
fig, ax = plt.subplots(figsize=(7, 4))
vals   = [err_expert_overall, err_train, err_deploy]
labels = ["Expert only\n(full run)", f"Fused TRAIN\n(k=0 to {K_CUT})",
          f"Mics DEPLOY\n(k={K_CUT+1} to end)"]
bars = ax.bar(labels, vals, width=0.5, color=["gray", "steelblue", "darkorange"])
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.3,
            f"{v:.2f}%", ha="center", fontsize=10)
ax.set_ylabel("Percent Error [%]")
ax.set_title(f"Multiclass Cutoff Effect (teacher={TEACHER_MODE})")
ax.set_ylim(0, max(vals) * 1.3)
plt.tight_layout()
plt.savefig("multiclass_cutoff_bar.png", dpi=150)
plt.show()

/tmp/ipykernel_993/1412503876.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/tmp/ipykernel_993/1412503876.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/tmp/ipykernel_993/1412503876.py:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# ==========================
# PLOT 4: Parameter convergence
# ==========================
fig, ax = plt.subplots(4, 1, sharex=True, figsize=(11, 10))

ax[0].plot(bh0_hist, color="steelblue", label=r"$\hat{b}_0(k)$ mic 1")
ax[0].plot(bh2_hist, color="darkorange", label=r"$\hat{b}_2(k)$ mic 2")
ax[0].axvline(K_CUT, linestyle="--", color="firebrick", label=f"CUT @ {K_CUT}")
ax[0].set_ylabel("Bias"); ax[0].legend(loc="upper right")
ax[0].set_title(f"Parameter convergence (teacher={TEACHER_MODE})")

ax[1].plot(ch0_hist, color="steelblue", label=r"$\hat{c}_0(k)$ mic 1")
ax[1].plot(ch2_hist, color="darkorange", label=r"$\hat{c}_2(k)$ mic 2")
ax[1].axvline(K_CUT, linestyle="--", color="firebrick")
ax[1].set_ylabel("Slope"); ax[1].legend(loc="upper right")

ax[2].plot(rh0_hist, color="steelblue", label=r"$\hat{\sigma}_0^2(k)$ mic 1")
ax[2].axvline(K_CUT, linestyle="--", color="firebrick")
ax[2].set_yscale("log"); ax[2].set_ylabel("Var mic 1"); ax[2].legend(loc="upper right")

ax[3].plot(rh2_hist, color="darkorange", label=r"$\hat{\sigma}_2^2(k)$ mic 2")
ax[3].axvline(K_CUT, linestyle="--", color="firebrick")
ax[3].set_yscale("log"); ax[3].set_ylabel("Var mic 2")
ax[3].set_xlabel("Time step k"); ax[3].legend(loc="upper right")

plt.tight_layout()
plt.savefig("multiclass_cutoff_params.png", dpi=150)
plt.show()

print("\nSaved figures:")
print("  multiclass_cutoff_confusion.png")
print("  multiclass_cutoff_overall.png")
print("  multiclass_cutoff_deploy.png")
print("  multiclass_cutoff_bar.png")
print("  multiclass_cutoff_params.png")


Saved figures:
  multiclass_cutoff_confusion.png
  multiclass_cutoff_overall.png
  multiclass_cutoff_deploy.png
  multiclass_cutoff_bar.png
  multiclass_cutoff_params.png


/tmp/ipykernel_993/1302253040.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Part B — Reporting for manuscript Section 4.4

Everything below reads `Xki_all`, `Xki_expert` and `Xd`. The filter is not
re-run and nothing above is modified.

In [9]:
# ---------- segment definitions ----------
TRAIN = slice(0, K_CUT + 1)
DEPLOY = slice(K_CUT + 1, Np)

def seg_err(pred, seg):
    return 100.0 * np.mean(pred[seg] != Xd[seg])

print("Percent error by segment")
print("=" * 60)
print(f"{'Condition':<30}{'Train':>10}{'Deploy':>10}{'Full':>10}")
print("-" * 60)
print(f"{'Expert alone':<30}"
      f"{seg_err(Xki_expert, TRAIN):>9.2f}%{seg_err(Xki_expert, DEPLOY):>9.2f}%"
      f"{100 * np.mean(Xki_expert != Xd):>9.2f}%")
print(f"{'Network':<30}"
      f"{seg_err(Xki_all, TRAIN):>9.2f}%{seg_err(Xki_all, DEPLOY):>9.2f}%"
      f"{100 * np.mean(Xki_all != Xd):>9.2f}%")
print("=" * 60)
print(f"Training samples 1-{K_CUT + 1}   |   Deployment samples {K_CUT + 2}-{Np}")

Percent error by segment
Condition                          Train    Deploy      Full
------------------------------------------------------------
Expert alone                      42.61%    46.49%    44.24%
Network                           36.29%    22.70%    30.56%
Training samples 1-1251   |   Deployment samples 1252-2163


### Segment comparability

The expert row above is the check the chapter asks for. The table below
reports what makes the two segments differ: how much of each class they
contain and how often the state changes. The counting task is the one where
this matters, because the deployment error is dominated by whichever class is
most common in that window.

In [10]:
print("Class distribution and transition count by segment")
print(f"{'Segment':<10}{'N':>7}{'0 persons':>16}{'1 person':>16}{'2 persons':>16}"
      f"{'Transitions':>13}")
for name, seg in [("Train", TRAIN), ("Deploy", DEPLOY), ("Full", slice(0, Np))]:
    x = Xd[seg]
    n_t = int(np.sum(np.abs(np.diff(x)) > 0))
    row = "".join(f"{np.sum(x == cl):>9} ({100 * np.mean(x == cl):>4.1f}%)"
                  for cl in [0, 1, 2])
    print(f"{name:<10}{len(x):>7}{row}{n_t:>13}")

Class distribution and transition count by segment
Segment         N       0 persons        1 person       2 persons  Transitions
Train        1251      528 (42.2%)      313 (25.0%)      410 (32.8%)           27
Deploy        912      332 (36.4%)      246 (27.0%)      334 (36.6%)           19
Full         2163      860 (39.8%)      559 (25.8%)      744 (34.4%)           46


In [11]:
# ---------- per-class recall ----------
def recall(pred, seg, cls):
    mask = (Xd[seg] == cls)
    return np.nan if mask.sum() == 0 else 100.0 * np.mean(pred[seg][mask] == cls)

print("Per-class recall [%]")
print(f"{'Condition':<28}{'0 persons':>12}{'1 person':>12}{'2 persons':>12}")
for name, pred, seg in [("Expert alone, train",  Xki_expert, TRAIN),
                        ("Expert alone, deploy", Xki_expert, DEPLOY),
                        ("Network, train",       Xki_all,    TRAIN),
                        ("Network, deploy",      Xki_all,    DEPLOY)]:
    print(f"{name:<28}" + "".join(f"{recall(pred, seg, cl):>12.2f}"
                                  for cl in [0, 1, 2]))

# Aggregate error is a class-weighted sum of the per-class error rates, so a
# segment holding more of an easy class will show a lower aggregate error
# without any change in the classifier itself.
print("\nDeployment error decomposition by class")
print(f"{'Class':<10}{'share':>9}{'class err':>12}{'contribution':>15}")
for cl in [0, 1, 2]:
    share = float(np.mean(Xd[DEPLOY] == cl))
    cerr = 100.0 - recall(Xki_all, DEPLOY, cl)
    print(f"{cl:<10}{share:>9.4f}{cerr:>11.2f}%{share * cerr:>14.2f}%")
print(f"{'total':<10}{1.0:>9.4f}{'':>12}{seg_err(Xki_all, DEPLOY):>14.2f}%")

Per-class recall [%]
Condition                      0 persons    1 person   2 persons
Expert alone, train               100.00       30.03       23.41
Expert alone, deploy              100.00       32.52       22.75
Network, train                    100.00       46.96       29.76
Network, deploy                    98.80       47.97       77.54

Deployment error decomposition by class
Class         share   class err   contribution
0            0.3640       1.20%          0.44%
1            0.2697      52.03%         14.04%
2            0.3662      22.46%          8.22%
total        1.0000                     22.70%


In [12]:
# ---------- Figure: per-class recall, matching Table 8 ----------
fig, ax = plt.subplots(figsize=(7.2, 3.8))
idx = np.arange(3)
w = 0.26
series = [("Expert alone, deployment", Xki_expert, DEPLOY, "#616161"),
          ("Network, training",        Xki_all,    TRAIN,  "steelblue"),
          ("Network, deployment",      Xki_all,    DEPLOY, "darkorange")]

for j, (label, pred, seg, colour) in enumerate(series):
    vals = [recall(pred, seg, cl) for cl in [0, 1, 2]]
    off = (j - 1) * w
    ax.bar(idx + off, vals, w, label=label, color=colour)
    for i, v in enumerate(vals):
        ax.text(i + off, v + 1.5, f"{v:.1f}", ha="center", fontsize=7)

ax.set_xticks(idx)
ax.set_xticklabels(["0 persons", "1 person", "2 persons"])
ax.set_ylabel("Recall [%]")
ax.set_ylim(0, 108)
ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3)
plt.tight_layout()
plt.savefig("ch5_counting_recall.pdf", bbox_inches="tight")
plt.show()

/tmp/ipykernel_993/3805178974.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
d = err_all[K_CUT+1:]
print(f"first 50 deploy samples : {100 * d[:50].mean():.2f}%  ({int(d[:50].sum())} errors)")
print(f"remaining {len(d)-50:>4} samples: {100 * d[50:].mean():.2f}%  ({int(d[50:].sum())} errors)")

first 50 deploy samples : 30.00%  (15 errors)
remaining  862 samples: 22.27%  (192 errors)


In [14]:
# ---------- paste-ready LaTeX ----------
teacher_label = {"gt": "Ground truth", "expert": "Expert-only",
                 "fused": "Fused"}[TEACHER_MODE]

print("% ---- Table 6 row ----")
print(f"    Person counting & ${Np}$ & ${K_CUT}$ & {teacher_label} \\\\")

print("\n% ---- Table 7: person counting across the cutoff ----")
print(r"  \begin{tabular}{@{}lccc@{}}")
print(r"    \toprule")
print(r"    Condition & Sensors in the filter & Samples & Error \\")
print(r"    \midrule")
print(f"    Expert alone, full recording & Expert only & $1$--${Np}$ & "
      f"${100 * np.mean(Xki_expert != Xd):.2f}\\%$ \\\\")
print(f"    Expert alone, deployment segment & Expert only & "
      f"${K_CUT + 2}$--${Np}$ & ${seg_err(Xki_expert, DEPLOY):.2f}\\%$ \\\\")
print(f"    Training, $k \\le k_c$ & Expert $+$ two inexpert & "
      f"$1$--${K_CUT + 1}$ & ${seg_err(Xki_all, TRAIN):.2f}\\%$ \\\\")
print(f"    Deployment, $k > k_c$ & Two inexpert only & "
      f"${K_CUT + 2}$--${Np}$ & ${seg_err(Xki_all, DEPLOY):.2f}\\%$ \\\\")
print(r"    \bottomrule")
print(r"  \end{tabular}")

print("\n% ---- Table: per-class recall across the cutoff ----")
print(r"  \begin{tabular}{@{}lccc@{}}")
print(r"    \toprule")
print(r"    Condition & $0$ persons & $1$ person & $2$ persons \\")
print(r"    \midrule")
for name, pred, seg in [("Expert alone, deployment segment", Xki_expert, DEPLOY),
                        ("Training, $k \\le k_c$",           Xki_all,    TRAIN),
                        ("Deployment, $k > k_c$",            Xki_all,    DEPLOY)]:
    vals = " & ".join(f"${recall(pred, seg, cl):.2f}\\%$" for cl in [0, 1, 2])
    print(f"    {name} & {vals} \\\\")
print(r"    \bottomrule")
print(r"  \end{tabular}")

print("\n% ---- class distribution of each segment ----")
print(r"  \begin{tabular}{@{}lcccc@{}}")
print(r"    \toprule")
print(r"    Segment & $N$ & $0$ persons & $1$ person & $2$ persons \\")
print(r"    \midrule")
for name, seg, lo, hi in [("Training", TRAIN, 1, K_CUT + 1),
                          ("Deployment", DEPLOY, K_CUT + 2, Np)]:
    x = Xd[seg]
    vals = " & ".join(f"${100 * np.mean(x == cl):.1f}\\%$" for cl in [0, 1, 2])
    print(f"    {name} (${lo}$--${hi}$) & ${len(x)}$ & {vals} \\\\")
print(r"    \bottomrule")
print(r"  \end{tabular}")

print("\n% ---- frozen parameters at k_c ----")
print(f"mic 1: b = {b_frozen_0:.4f}, c = {c_frozen_0:.4f}, "
      f"sigma^2 = {sig2_frozen_0:.4f}")
print(f"mic 2: b = {b_frozen_2:.4f}, c = {c_frozen_2:.4f}, "
      f"sigma^2 = {sig2_frozen_2:.4f}")

% ---- Table 6 row ----
    Person counting & $2163$ & $1250$ & Expert-only \\

% ---- Table 7: person counting across the cutoff ----
  \begin{tabular}{@{}lccc@{}}
    \toprule
    Condition & Sensors in the filter & Samples & Error \\
    \midrule
    Expert alone, full recording & Expert only & $1$--$2163$ & $44.24\%$ \\
    Expert alone, deployment segment & Expert only & $1252$--$2163$ & $46.49\%$ \\
    Training, $k \le k_c$ & Expert $+$ two inexpert & $1$--$1251$ & $36.29\%$ \\
    Deployment, $k > k_c$ & Two inexpert only & $1252$--$2163$ & $22.70\%$ \\
    \bottomrule
  \end{tabular}

% ---- Table: per-class recall across the cutoff ----
  \begin{tabular}{@{}lccc@{}}
    \toprule
    Condition & $0$ persons & $1$ person & $2$ persons \\
    \midrule
    Expert alone, deployment segment & $100.00\%$ & $32.52\%$ & $22.75\%$ \\
    Training, $k \le k_c$ & $100.00\%$ & $46.96\%$ & $29.76\%$ \\
    Deployment, $k > k_c$ & $98.80\%$ & $47.97\%$ & $77.54\%$ \\
    \bottomrule
  \end{